In [1]:
# Uncomment if not installed
# !pip install mlflow scikit-learn pandas

import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)

print('✅ All libraries imported!')
print(f'📌 MLflow version: {mlflow.__version__}')

✅ All libraries imported!
📌 MLflow version: 3.10.0


In [2]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

print('📦 Dataset Info:')
print(f'   Total Samples  : {len(X)}')
print(f'   Features       : {X.shape[1]}')
print(f'   Target Classes : {list(data.target_names)}')
print(f'   Class Balance  : {dict(y.value_counts())}')

X.head(3)

📦 Dataset Info:
   Total Samples  : 569
   Features       : 30
   Target Classes : [np.str_('malignant'), np.str_('benign')]
   Class Balance  : {1: np.int64(357), 0: np.int64(212)}


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.0,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y        # Keeps class balance in both splits
)

print(f'✅ Train size : {len(X_train)}')
print(f'✅ Test size  : {len(X_test)}')

✅ Train size : 455
✅ Test size  : 114


In [4]:
def evaluate_model(model, X_test, y_test):
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    return {
        'accuracy'  : round(accuracy_score(y_test, y_pred), 4),
        'precision' : round(precision_score(y_test, y_pred), 4),
        'recall'    : round(recall_score(y_test, y_pred), 4),
        'f1_score'  : round(f1_score(y_test, y_pred), 4),
        'roc_auc'   : round(roc_auc_score(y_test, y_proba), 4),
    }

print('✅ evaluate_model() is ready!')

✅ evaluate_model() is ready!


In [5]:
# Each dict = one MLflow run
# We go from weak → moderate → strong → overfit → regularized → no bootstrap
# So we can clearly see the impact of each change in the MLflow UI

hyperparameter_grid = [

    {   # Run 1 — Intentionally weak baseline
        'run_name'          : 'Baseline_Weak',
        'n_estimators'      : 10,
        'max_depth'         : 2,
        'min_samples_split' : 2,
        'max_features'      : 'sqrt',
        'bootstrap'         : True,
    },
    {   # Run 2 — Moderate
        'run_name'          : 'Moderate',
        'n_estimators'      : 50,
        'max_depth'         : 5,
        'min_samples_split' : 5,
        'max_features'      : 'sqrt',
        'bootstrap'         : True,
    },
    {   # Run 3 — Strong
        'run_name'          : 'Strong',
        'n_estimators'      : 100,
        'max_depth'         : 10,
        'min_samples_split' : 2,
        'max_features'      : 'sqrt',
        'bootstrap'         : True,
    },
    {   # Run 4 — Deep, no depth limit → overfitting risk
        'run_name'          : 'Deep_Overfit_Risk',
        'n_estimators'      : 200,
        'max_depth'         : None,
        'min_samples_split' : 2,
        'max_features'      : 'sqrt',
        'bootstrap'         : True,
    },
    {   # Run 5 — Regularized (more controlled)
        'run_name'          : 'Regularized',
        'n_estimators'      : 150,
        'max_depth'         : 8,
        'min_samples_split' : 10,
        'max_features'      : 'log2',
        'bootstrap'         : True,
    },
    {   # Run 6 — No Bootstrap
        'run_name'          : 'No_Bootstrap',
        'n_estimators'      : 100,
        'max_depth'         : 8,
        'min_samples_split' : 5,
        'max_features'      : 'sqrt',
        'bootstrap'         : False,
    },
]

print(f'✅ {len(hyperparameter_grid)} configurations ready!')
print('Runs:', [c["run_name"] for c in hyperparameter_grid])

✅ 6 configurations ready!
Runs: ['Baseline_Weak', 'Moderate', 'Strong', 'Deep_Overfit_Risk', 'Regularized', 'No_Bootstrap']


In [8]:
EXPERIMENT_NAME = 'RandomForest_HyperParam_Tuning'

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment(EXPERIMENT_NAME)  # Creates experiment if not exists

print(f'✅ Experiment set: "{EXPERIMENT_NAME}"')
print('📁 Logs will be saved in: ./mlruns/')

2026/03/02 11:45:43 INFO mlflow.tracking.fluent: Experiment with name 'RandomForest_HyperParam_Tuning' does not exist. Creating a new experiment.


✅ Experiment set: "RandomForest_HyperParam_Tuning"
📁 Logs will be saved in: ./mlruns/


In [9]:
results_summary = []

for config in hyperparameter_grid:

    run_name = config['run_name']
    params   = {k: v for k, v in config.items() if k != 'run_name'}

    with mlflow.start_run(run_name=run_name):

        # Train
        model = RandomForestClassifier(random_state=42, **params)
        model.fit(X_train, y_train)

        # Evaluate
        metrics = evaluate_model(model, X_test, y_test)

        # Log hyperparameters → shows in "Parameters" column in UI
        mlflow.log_params(params)

        # Log metrics → shows in "Metrics" column in UI
        mlflow.log_metrics(metrics)

        # Save the actual model as an artifact
        mlflow.sklearn.log_model(model, artifact_path='model')

        # Tags → extra metadata
        mlflow.set_tags({'dataset': 'breast_cancer', 'algorithm': 'RandomForest'})

        print(f'\n✅ Run: {run_name}')
        print(f'   n_estimators={params["n_estimators"]} | max_depth={params["max_depth"]} | min_samples_split={params["min_samples_split"]}')
        print(f'   Accuracy={metrics["accuracy"]} | F1={metrics["f1_score"]} | AUC={metrics["roc_auc"]}')

        results_summary.append({'Run': run_name, **metrics})

print('\n🎉 All runs logged to MLflow!')

2026/03/02 11:46:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/02 11:46:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



✅ Run: Baseline_Weak
   n_estimators=10 | max_depth=2 | min_samples_split=2
   Accuracy=0.9386 | F1=0.951 | AUC=0.9897
🏃 View run Baseline_Weak at: http://127.0.0.1:5000/#/experiments/7/runs/5c8a5ff1e3fc4c13a14ee54c46ae9788
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


2026/03/02 11:46:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/02 11:46:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



✅ Run: Moderate
   n_estimators=50 | max_depth=5 | min_samples_split=5
   Accuracy=0.9561 | F1=0.9655 | AUC=0.9937
🏃 View run Moderate at: http://127.0.0.1:5000/#/experiments/7/runs/5640b1a3396543cab2c63fdf5290dc7c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


2026/03/02 11:46:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/02 11:46:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...


✅ Run: Deep_Overfit_Risk
   n_estimators=200 | max_depth=None | min_samples_split=2
   Accuracy=0.9561 | F1=0.9655 | AUC=0.9931
🏃 View run Deep_Overfit_Risk at: http://127.0.0.1:5000/#/experiments/7/runs/43e9da58e4e54e9d903119bd0d44d4a2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


2026/03/02 11:46:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/02 11:46:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



✅ Run: Regularized
   n_estimators=150 | max_depth=8 | min_samples_split=10
   Accuracy=0.9474 | F1=0.9583 | AUC=0.9917
🏃 View run Regularized at: http://127.0.0.1:5000/#/experiments/7/runs/9522ab1a71e1449b9bc0a492c8baa6f7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


2026/03/02 11:46:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/02 11:46:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



✅ Run: No_Bootstrap
   n_estimators=100 | max_depth=8 | min_samples_split=5
   Accuracy=0.9561 | F1=0.9655 | AUC=0.9934
🏃 View run No_Bootstrap at: http://127.0.0.1:5000/#/experiments/7/runs/fb25c9fcfb324dd0a68237c085dbac6c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7

🎉 All runs logged to MLflow!


In [10]:
df_results = pd.DataFrame(results_summary)
df_results = df_results.sort_values('f1_score', ascending=False).reset_index(drop=True)
df_results.index += 1
df_results.index.name = 'Rank'

print('📊 Results sorted by F1 Score (Best → Worst):\n')
df_results

📊 Results sorted by F1 Score (Best → Worst):



,Run,accuracy,precision,recall,f1_score,roc_auc
Rank,,,,,,
1,Moderate,0.9561,0.9589,0.9722,0.9655,0.9937
2,Strong,0.9561,0.9589,0.9722,0.9655,0.9937
3,No_Bootstrap,0.9561,0.9589,0.9722,0.9655,0.9934
4,Deep_Overfit_Risk,0.9561,0.9589,0.9722,0.9655,0.9931
5,Regularized,0.9474,0.9583,0.9583,0.9583,0.9917
6,Baseline_Weak,0.9386,0.9577,0.9444,0.9510,0.9897


In [11]:
best  = df_results.iloc[0]
worst = df_results.iloc[-1]

print('🏆 Best Run:')
print(f'   Name     : {best["Run"]}')
print(f'   Accuracy : {best["accuracy"]}')
print(f'   F1 Score : {best["f1_score"]}')
print(f'   ROC AUC  : {best["roc_auc"]}')

print('\n⚠️  Worst Run:')
print(f'   Name     : {worst["Run"]}')
print(f'   F1 Score : {worst["f1_score"]}')

diff = round(best['f1_score'] - worst['f1_score'], 4)
print(f'\n📈 F1 improvement (worst → best): +{diff}')
print('   This is the power of hyperparameter tuning!')

🏆 Best Run:
   Name     : Moderate
   Accuracy : 0.9561
   F1 Score : 0.9655
   ROC AUC  : 0.9937

⚠️  Worst Run:
   Name     : Baseline_Weak
   F1 Score : 0.951

📈 F1 improvement (worst → best): +0.0145
   This is the power of hyperparameter tuning!


In [12]:
client     = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name('RandomForest_HyperParam_Tuning')

runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.mlflow.runName = '{best['Run']}'",
    order_by=['metrics.f1_score DESC'],
    max_results=1
)

best_run_id = runs[0].info.run_id
print(f'✅ Best Run ID: {best_run_id}')

# Load model back from MLflow
loaded_model = mlflow.sklearn.load_model(f'runs:/{best_run_id}/model')

# Predict on 5 test samples
predictions = loaded_model.predict(X_test.iloc[:5])
actuals     = y_test.iloc[:5].values

print('\n🔮 Predictions vs Actuals:')
for i, (pred, act) in enumerate(zip(predictions, actuals)):
    match = '✅' if pred == act else '❌'
    print(f'   Sample {i+1}: Predicted={data.target_names[pred]:<12} | Actual={data.target_names[act]} {match}')

✅ Best Run ID: 5640b1a3396543cab2c63fdf5290dc7c

🔮 Predictions vs Actuals:
   Sample 1: Predicted=malignant    | Actual=malignant ✅
   Sample 2: Predicted=benign       | Actual=benign ✅
   Sample 3: Predicted=malignant    | Actual=malignant ✅
   Sample 4: Predicted=malignant    | Actual=benign ❌
   Sample 5: Predicted=malignant    | Actual=malignant ✅
